# Notebook 2 (Fixed): Static Baseline — Experiment 1
**IEEE IES GenAI Challenge 2026**

Key fixes over previous version:
- **Model:** `llama-3.3-70b-versatile` (8B is too weak for feature-level causal attribution)
- **Few-shot:** 3 examples per class (12 total) — necessary for inter-class discrimination
- **Rate limit handling:** persistent checkpoint cache, adaptive backoff, configurable batch size
- **Resume support:** re-run the cell to continue from where you left off

**Groq free tier limits:** ~30 RPM, ~6000 tokens/min for 70B. With 12 few-shot examples
each prompt is ~1800 tokens. Safe rate: 1 request per 3 seconds = 20 RPM.

In [ ]:
!pip install groq scikit-learn tqdm -q

In [ ]:
import os, pickle, time, json, re
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from groq import Groq, RateLimitError
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print('Dependencies loaded.')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
GROQ_API_KEY   = 'your_groq_api_key_here'

MODEL          = 'llama-3.3-70b-versatile'  # DO NOT change — 8B fails this task
MAX_TOKENS     = 400
TEMPERATURE    = 0.0

# Rate limit settings — tune if you hit 429s
BASE_DELAY     = 3.0   # seconds between requests (conservative for 70B)
MAX_BACKOFF    = 120   # max seconds to wait on rate limit error
MAX_RETRIES    = 6

LABEL_MAP    = {0: 'Normal', 1: 'Inner Race', 2: 'Outer Race', 3: 'Ball'}
NAME_TO_ID   = {v: k for k, v in LABEL_MAP.items()}
VALID_LABELS = list(LABEL_MAP.values())

CACHE_PATH = Path('baseline_cache.pkl')  # persistent — survives Colab disconnects

In [ ]:
with open('cwru_features.pkl', 'rb') as f:
    data = pickle.load(f)

df_test          = data['df_test']
fewshot_examples = data['fewshot_examples']  # 12 examples: 3 per class

print(f'Test samples:      {len(df_test)}')
print(f'Few-shot examples: {len(fewshot_examples)} ({len(fewshot_examples)//4} per class)')
print(f'Model:             {MODEL}')

## 1. Prompts

**Why 3 examples per class matters:** with only 1 example per class, the model can memorize a single feature pattern per fault type. With 3, it sees feature variance within each class and learns to discriminate based on the *distribution* of features rather than a single threshold.

**Why the system prompt includes explicit thresholds:** prevents the 8B failure mode where the model calls kurtosis=-0.11 "high". The thresholds anchor the model's reasoning to physical reality.

In [ ]:
SYSTEM_PROMPT = """You are an expert in rotating machinery fault diagnosis. Perform causal attribution of bearing faults from vibration signal features.

Classify into exactly ONE category with physical reasoning:
- Normal: kurtosis < 3, crest factor < 3, all band energies low
- Inner Race: kurtosis > 4, crest factor > 3.5, BPFI band energy highest
- Outer Race: kurtosis 3-5, BPFO band energy highest or dominant
- Ball: variable kurtosis, BSF band energy highest, often highest crest factor

Decision logic:
1. Check kurtosis and crest factor to assess impulsiveness
2. Compare BPFI vs BPFO vs BSF band energies — highest band energy guides fault type
3. Normal if all indicators are low

Respond ONLY with valid JSON, no preamble:
{"fault_type": "<Normal|Inner Race|Outer Race|Ball>", "reasoning": "<2-sentence causal explanation referencing specific values>", "confidence": <0.0-1.0>}"""


def build_fewshot_block(examples):
    """Use all 12 examples (3 per class). Order: Normal, Inner Race, Outer Race, Ball."""
    # Sort by class to ensure balanced ordering
    sorted_examples = sorted(examples, key=lambda x: x['label'])
    block = ''
    for ex in sorted_examples:
        block += f'\n--- EXAMPLE ({ex["label_name"]}) ---\n'
        block += f'Signal:\n{ex["text"]}\n'
        block += f'Diagnosis: {{"fault_type": "{ex["label_name"]}", "reasoning": "Verified {ex["label_name"]} fault.", "confidence": 0.95}}\n'
    return block


FEWSHOT_BLOCK = build_fewshot_block(fewshot_examples)

def build_user_message(signal_text):
    return f'{FEWSHOT_BLOCK}\n--- QUERY ---\nSignal:\n{signal_text}\nDiagnosis:'


def parse_response(text):
    try:
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
            ft = parsed.get('fault_type', '').strip()
            # Normalize common LLM variations
            ft = ft.replace('inner race', 'Inner Race').replace('outer race', 'Outer Race')
            ft = ft.replace('ball fault', 'Ball').replace('normal', 'Normal')
            if ft in VALID_LABELS:
                return ft, parsed.get('reasoning', ''), float(parsed.get('confidence', 0.5))
    except Exception:
        pass
    # Fuzzy fallback
    text_lower = text.lower()
    for label in VALID_LABELS:
        if label.lower() in text_lower:
            return label, text[:200], 0.4
    return None, text[:200], 0.0


print('Prompts built.')
est_tokens_per_prompt = (len(SYSTEM_PROMPT) + len(FEWSHOT_BLOCK)) // 4 + 300
print(f'Estimated tokens per prompt: ~{est_tokens_per_prompt}')
print(f'Estimated total tokens (200 samples): ~{est_tokens_per_prompt * 200:,}')
print(f'Estimated runtime at {BASE_DELAY}s/req: ~{200 * BASE_DELAY / 60:.1f} minutes')

## 2. Inference with Rate-Limit-Aware Retry Logic

**Checkpoint cache:** results are saved every 5 samples. If Colab disconnects or you hit a hard rate limit, re-run this cell and it will resume from the last saved sample.

In [ ]:
client = Groq(api_key=GROQ_API_KEY)

def classify_with_retry(signal_text):
    """Call Groq with exponential backoff on rate limit errors."""
    delay = BASE_DELAY
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': build_user_message(signal_text)}
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            )
            raw = response.choices[0].message.content
            time.sleep(BASE_DELAY)  # polite delay after success
            return parse_response(raw)

        except RateLimitError as e:
            # Extract retry-after header if present
            wait = min(delay * (2 ** attempt), MAX_BACKOFF)
            print(f'  Rate limit hit. Waiting {wait:.0f}s... (attempt {attempt+1}/{MAX_RETRIES})')
            time.sleep(wait)

        except Exception as e:
            wait = min(delay * (2 ** attempt), 30)
            print(f'  API error: {e}. Waiting {wait:.0f}s...')
            time.sleep(wait)

    return None, 'Max retries exceeded', 0.0


# ── Load checkpoint ───────────────────────────────────────────────────────────
if CACHE_PATH.exists():
    with open(CACHE_PATH, 'rb') as f:
        cache = pickle.load(f)
    print(f'Resuming from checkpoint: {len(cache)} / {len(df_test)} samples done.')
else:
    cache = {}
    print('Starting fresh.')

remaining = [i for i in df_test.index if i not in cache]
print(f'Remaining: {len(remaining)} samples | ETA: ~{len(remaining) * BASE_DELAY / 60:.1f} min')


# ── Run inference ─────────────────────────────────────────────────────────────
for idx in tqdm(remaining, desc=f'Baseline ({MODEL})'):
    row = df_test.loc[idx]
    ft, reasoning, conf = classify_with_retry(row['text'])
    cache[idx] = {
        'pred_name':  ft,
        'pred_id':    NAME_TO_ID.get(ft, -1) if ft else -1,
        'reasoning':  reasoning,
        'confidence': conf,
        'true_id':    row['label'],
        'true_name':  row['label_name'],
    }
    # Save checkpoint every 5 samples
    if len(cache) % 5 == 0:
        with open(CACHE_PATH, 'wb') as f:
            pickle.dump(cache, f)

# Final save
with open(CACHE_PATH, 'wb') as f:
    pickle.dump(cache, f)
print(f'\nDone. Total cached: {len(cache)} samples.')

## 3. Evaluate

In [ ]:
records = list(cache.values())
valid   = [r for r in records if r['pred_id'] != -1]
failed  = len(records) - len(valid)

true_ids = [r['true_id'] for r in valid]
pred_ids = [r['pred_id'] for r in valid]

accuracy    = accuracy_score(true_ids, pred_ids)
f1_macro    = f1_score(true_ids, pred_ids, average='macro')
f1_per_cls  = f1_score(true_ids, pred_ids, average=None, labels=[0,1,2,3])
cm          = confusion_matrix(true_ids, pred_ids, labels=[0,1,2,3])
target_names = [LABEL_MAP[i] for i in range(4)]

if failed:
    print(f'WARNING: {failed} failed/unparseable predictions excluded.')

print('=' * 60)
print('EXPERIMENT 1: STATIC BASELINE RESULTS')
print(f'Model: {MODEL}')
print('=' * 60)
print(f'Accuracy (overall):  {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'F1 Score (macro):    {f1_macro:.4f}')
print()
print('F1 per class:')
for cid in range(4):
    print(f'  {LABEL_MAP[cid]:12s}: {f1_per_cls[cid]:.4f}')
print()
print(classification_report(true_ids, pred_ids, target_names=target_names, labels=[0,1,2,3]))
print('Confusion Matrix (rows=true, cols=pred):')
print(pd.DataFrame(cm, index=target_names, columns=target_names))

# Sanity check: flag if model is still collapsing to one class
from collections import Counter
pred_dist = Counter(pred_ids)
print(f'\nPrediction distribution: {dict(pred_dist)}')
if max(pred_dist.values()) > len(valid) * 0.6:
    dominant = LABEL_MAP[max(pred_dist, key=pred_dist.get)]
    print(f'WARNING: Model is collapsing — over 60% predictions are {dominant}.')
    print('This usually means the model or few-shot examples need adjustment.')
else:
    print('Prediction distribution looks reasonable.')

In [ ]:
# Inspect reasoning chains across all 4 predicted classes
print('=== SAMPLE REASONING CHAINS BY PREDICTED CLASS ===')
shown = set()
for r in records:
    pname = r['pred_name']
    if pname and pname not in shown:
        correct = '✓' if r['true_name'] == pname else '✗'
        print(f'\n[{correct}] True: {r["true_name"]:12s} | Predicted: {pname:12s} | Conf: {r["confidence"]:.2f}')
        print(f'Reasoning: {r["reasoning"][:300]}')
        shown.add(pname)
    if len(shown) == 4:
        break

In [ ]:
# Save for Notebook 3
baseline_results = {
    'predictions':      pred_ids,
    'true_labels':      true_ids,
    'reasonings':       [r['reasoning'] for r in valid],
    'confidences':      [r['confidence'] for r in valid],
    'accuracy':         accuracy,
    'f1_macro':         f1_macro,
    'f1_per_class':     f1_per_cls,
    'confusion_matrix': cm,
    'model':            MODEL,
    'n_samples':        len(valid),
    'n_failed':         failed,
}

with open('baseline_results.pkl', 'wb') as f:
    pickle.dump(baseline_results, f)

print('Saved baseline_results.pkl')
print('Ready for Notebook 3 (TTC attribution loop).')